# Projet — Détection de faux profils

**Comment travailler :** exécute les cellules **dans l’ordre** (Menu *Run → Run All Above* si tu es perdu·e).

**Avant la 1ère fois :** dans un terminal à la racine du projet, lancer :
`python scripts/make_messy_dataset.py`

**Important :** ouvrir Jupyter **depuis la racine** du dossier `fakes-account`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.paths import DATA_RAW, DATA_PROCESSED, MODELS_DIR
from src.data_loading import load_csv

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("OK — Racine projet :", ROOT)
print("Données brutes :", DATA_RAW)

## 1 — Charger les données (version « à nettoyer »)

Si une erreur apparaît, exécute d’abord `python scripts/make_messy_dataset.py` dans un terminal.

In [ ]:
import pandas as pd

df = load_csv("fake_social_media_messy.csv")
df.head()

### 1.1 Forme du tableau

In [ ]:
print("Nombre de lignes, colonnes :", df.shape)
print("\nColonnes :", df.columns.tolist())
df.dtypes

### 1.2 Valeurs manquantes et doublons

In [ ]:
manquants = df.isna().sum()
print("Total NaN par colonne :")
print(manquants[manquants > 0])
print("\nTotal de cellules NaN :", df.isna().sum().sum())

dups = df.duplicated().sum()
print("\nLignes en double (toutes colonnes) :", dups)

### 1.3 Cible `is_fake` et variable `platform`

In [ ]:
print(df["is_fake"].value_counts(dropna=False))
print("\n", df["platform"].value_counts(dropna=False))

### 1.4 Quelques graphiques (optionnel mais bien pour le rapport)

Si `ModuleNotFoundError: matplotlib`, installe dans le venv : `pip install matplotlib seaborn`.

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df["followers"].dropna().hist(bins=40, ax=axes[0])
axes[0].set_title("Followers (sans NaN)")
df["is_fake"].value_counts().plot(kind="bar", ax=axes[1], rot=0)
axes[1].set_title("Répartition is_fake")
plt.tight_layout()
plt.show()

---
## 2 — Premier nettoyage (à adapter / compléter)

Voici une base **simple et classique** : tu pourras modifier les choix et les expliquer dans ton rapport.

In [ ]:
df_clean = df.copy()

# Supprimer les doublons exacts
avant = len(df_clean)
df_clean = df_clean.drop_duplicates()
print("Lignes supprimées (doublons) :", avant - len(df_clean))

# Harmoniser les noms de plateforme
plat = df_clean["platform"].astype(str).str.strip().str.lower()
mapping = {"instagram": "Instagram", "facebook": "Facebook", "twitter": "Twitter"}
df_clean["platform"] = plat.map(mapping).fillna(df_clean["platform"])
print("\nPlateformes après normalisation :")
print(df_clean["platform"].value_counts())

In [ ]:
# Imputation : médiane pour les colonnes numériques (sauf la cible)
cols_num = df_clean.select_dtypes(include="number").columns.drop("is_fake", errors="ignore")
for col in cols_num:
    med = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(med)

print("NaN restants :", df_clean.isna().sum().sum())

In [ ]:
# Outliers grossiers sur followers (exemple : winsorisation simple au 99e percentile)
seuil = df_clean["followers"].quantile(0.99)
nb_clip = (df_clean["followers"] > seuil).sum()
df_clean["followers"] = df_clean["followers"].clip(upper=seuil)
print(f"Followers plafonnés au percentile 99 (~{seuil:.0f}), lignes concernées : {nb_clip}")

# Encodage one-hot de platform (évite d’injecter du texte brut dans sklearn)
df_model = pd.get_dummies(df_clean, columns=["platform"], drop_first=False)
df_model.head()

In [ ]:
chemin_propre = DATA_PROCESSED / "profiles_clean.csv"
df_model.to_csv(chemin_propre, index=False)
print("Fichier sauvegardé :", chemin_propre)

---
## 3 — Entraînement des modèles

**À faire ensuite :** `train_test_split` sur `df_model`, séparer `X` et `y` (`is_fake`), entraîner régression logistique, KNN, arbre, Random Forest, SVM, Naïve Bayes, sauvegarder avec `joblib` dans `models/`.

In [ ]:
# Exemple minimal (à étendre avec tous les modèles du sujet)
# from sklearn.model_selection import train_test_split
# X = df_model.drop(columns=["is_fake"])
# y = df_model["is_fake"]
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# ...
print("Complète cette section une fois le nettoyage validé.")

## 4 — Évaluation et comparaison

**À faire :** `classification_report`, `confusion_matrix`, comparer les modèles pour le rapport et le frontend.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("Exemple d’usage après y_pred :")
print(classification_report.__doc__.split("\n")[0])